# Rating Prediction - Portuguese Product Reviews
**Features selecionadas do artigo BRACIS 2021 + features adicionais + titulo separado**

### Features do artigo utilizadas
- **POS (Tabela 1):** Feats 1 (ADJ), 3 (ADV), 6 (DET), 7 (INTJ), 9 (NUM), 12 (PUNCT), 14 (SYM), 16 (COMP), 17 (SUP), 18 (X)
- **Padroes sintaticos (Tabela 2):** Feats 19-23
- **Lexico de sentimento (Tabela 3):** Feats 24-31
- **Tabela 4 (concept-level):** ignorada
- **Estrutural:** Feat 46 (uppercase_ratio)
- **Twitter (Tabela 6):** Feats 47-51

### Features extras (nao tratadas no artigo)
- Negacao invertendo polaridade (`neg_before_pos`, `neg_before_neg`) - janela de 3 tokens
- Intensificadores e atenuadores antes de adjetivos
- Palavras ALL CAPS (emocao forte)
- Exclamacoes e interrogacoes separadas
- Ratio negativo/positivo
- Verbos na primeira pessoa
- Type-token ratio (diversidade lexical)
- Comprimento do texto

### Titulo analisado separadamente
Cada feature e calculada independentemente para `text` e `title`.
Features do titulo recebem prefixo `title_`.
Isso preserva a funcao sintatica de cada campo - um adjetivo do titulo nao interfere
no papel dos tokens do texto.


## 1. Imports & Configuracao

In [ ]:
import re, warnings, numpy as np, pandas as pd
from pathlib import Path
import spacy
from spacy.tokens.doc import Doc
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from scipy import stats

warnings.filterwarnings('ignore')

current_dir = Path(globals().get('_dh', ['.'])[0])
csv_folder  = current_dir / 'csv'
print('Bibliotecas carregadas.')


Bibliotecas carregadas.


## 2. Carregar Dados

In [ ]:
train_df = pd.read_csv(csv_folder / 'train.csv', usecols=['id', 'text', 'title', 'rating'])
test_df  = pd.read_csv(csv_folder / 'test.csv',  usecols=['id', 'text', 'title'])

# Garantir que text e title sao strings (sem NaN)
for df in [train_df, test_df]:
    df['text']  = df['text'].fillna('').astype(str)
    df['title'] = df['title'].fillna('').astype(str)

print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')
print('\nDistribuicao de ratings:')
print(train_df['rating'].value_counts().sort_index())
print('\nExemplo de texto :', train_df['text'].iloc[0][:80])
print('Exemplo de titulo:', train_df['title'].iloc[0][:80])


Train: 41,005 | Test: 10,252

Distribuicao de ratings:
rating
1    8205
2    8178
3    8194
4    8198
5    8230
Name: count, dtype: int64

Exemplo de texto : Por incrível que pareça, para uma bebida destilada, de alta concentração alcoóli
Exemplo de titulo: Vodka diferenciada


## 3. Carregar Modelo spaCy

In [ ]:
nlp = spacy.load('pt_core_news_lg')
print(f'spaCy: {nlp.meta["name"]} v{nlp.meta["version"]}')


spaCy: core_news_lg v3.8.0


## 4. Lexicos de Sentimento

Cobre as **Features 24-31** do artigo (Tabela 3).
Palavras com e sem acentuacao para maxima cobertura.

In [ ]:
# Termos positivos (Features 25, 29)
POSITIVE_TERMS = {
    'bom','boa','bons','boas',
    'otimo','otima','otimos','otimas',
    'ótimo','ótima','ótimos','ótimas',
    'excelente','excelentes',
    'maravilhoso','maravilhosa','maravilhosos','maravilhosas',
    'perfeito','perfeita','perfeitos','perfeitas',
    'incrivel','incriveis','incrível','incríveis',
    'fantastico','fantastica','fantástico','fantástica','fantásticos','fantásticas',
    'sensacional','sensacionais',
    'lindo','linda','lindos','lindas',
    'bonito','bonita','bonitos','bonitas',
    'agradavel','agradaveis','agradável','agradáveis',
    'delicioso','deliciosa','deliciosos','deliciosas',
    'gostoso','gostosa','gostosos','gostosas',
    'rapido','rapida','rapidos','rapidas','rápido','rápida','rápidos','rápidas',
    'eficiente','eficientes','resistente','resistentes',
    'duravel','duraveis','durável','duráveis',
    'confortavel','confortaveis','confortável','confortáveis',
    'pratico','pratica','praticos','praticas','prático','prática','práticos','práticas',
    'funcional','funcionais','satisfeito','satisfeita','satisfeitos','satisfeitas',
    'feliz','felizes','top','show',
    'adorei','amei','gostei','aprovei','recomendo','recomendei',
    'superou','surpreendeu','encantou','impressionou',
    'funcionou','funcionando','chegou','entregou',
    'vale','valeu','valendo','curti','curtei','adoramos','amamos',
    'qualidade','excelência','perfeição','satisfação','prazer','amor',
}

# Termos negativos (Features 26, 30)
NEGATIVE_TERMS = {
    'ruim','ruins',
    'pessimo','pessima','pessimos','pessimas','péssimo','péssima','péssimos','péssimas',
    'horrivel','horriveis','horrível','horríveis',
    'terrivel','terriveis','terrível','terríveis',
    'lixo','lixos','porcaria',
    'fraco','fraca','fracos','fracas',
    'quebrado','quebrada','quebrados','quebradas',
    'defeituoso','defeituosa','defeituosos','defeituosas',
    'demorado','demorada','demorados','demoradas',
    'caro','cara','caros','caras',
    'decepcionante','decepcionantes',
    'insatisfatorio','insatisfatoria','insatisfatório','insatisfatória',
    'inadequado','inadequada','inadequados','inadequadas',
    'falso','falsa','falsos','falsas',
    'inutil','inuteis','inútil','inúteis',
    'dificil','dificeis','difícil','difíceis',
    'desconfortavel','desconfortaveis','desconfortável','desconfortáveis',
    'fragil','frágil','frageis','frágeis','horroroso','horrorosa',
    'odiei','detestei','decepcionou','frustrou',
    'quebrou','parou','travou','bugou','emperrou',
    'arrependido','arrependida','arrependei',
    'devolvendo','devolvi','cancelei','reclamei','vazou','estragou','queimou',
    'decepcao','decepção','vergonha','fraude','golpe',
    'problema','problemas','defeito','defeitos',
    'absurdo','absurdos','reclamacao','reclamação','prejuizo','prejuízo',
}

# Marcadores de opiniao (Feature 27)
OPINION_MARKERS = {
    'acho','achei','acredito','acreditei','penso','pensei','considero','considerei','creio',
    'parece','parecer','pareceu','aparentemente',
    'sinto','senti','sentindo','espero','esperava','esperei',
    'recomendo','recomendei','recomendaria','indicaria',
    'vale','valeu','pra mim','para mim','no meu ver',
}

SUBJECTIVE_TERMS = POSITIVE_TERMS | NEGATIVE_TERMS | OPINION_MARKERS

# Comparativos e superlativos (Features 16, 17)
COMP_LIST = ['melhor','pior','maior','menor','mais','menos']
SUP_LIST  = [
    'otimo','otima','otimos','otimas','ótimo','ótima','ótimos','ótimas',
    'pessimo','pessima','pessimos','pessimas','péssimo','péssima','péssimos','péssimas',
    'maximo','maxima','minimo','minima','máximo','máxima','mínimo','mínima',
    'excelente','excelentes','horrivel','horriveis','horrível','horríveis',
    'terrivel','terriveis','terrível','terríveis',
]

# Negacao
NEGATION_WORDS = {
    'nao','não','nem','nunca','jamais','tampouco',
    'nenhum','nenhuma','nenhuns','nenhumas',
}

# Intensificadores
INTENSIFIERS = {
    'muito','muita','muitos','muitas','bastante','bastantes','demais',
    'extremamente','super','mega','ultra','hiper','bem',
    'incrivelmente','absurdamente','completamente','totalmente','realmente',
    'tão','tao','verdadeiramente',
}

# Atenuadores
ATTENUATORS = {
    'pouco','pouca','poucos','poucas',
    'meio','meia','quase','levemente','ligeiramente',
    'razoavelmente','mediamente',
}

# Verbos de primeira pessoa
FIRST_PERSON_VERBS = {
    'acho','achei','acredito','acreditei','penso','pensei',
    'sinto','senti','espero','esperava',
    'recomendo','recomendei','indico','indiquei',
    'comprei','recebi','usei','testei','experimentei',
    'adorei','amei','odiei','detestei','gostei','curtei',
    'devolvi','reclamei','cancelei','pedi',
    'vou','fui','vim','tenho','tive',
}

# Emoticons (Feature 51)
EMOTICON_POLARITY = {
    ':)':1,':-)'  :1,':D':1,':-D':1,'=)':1,'=D':1,
    ';)':1,';-)' :1,':P':1,':-P':1,':p':1,':-p':1,
    '<3':1,':*'  :1,':-*':1,'^_^':1,'^-^':1,":')": 1,
    ':(':  -1,':-(':-1,":'(":-1,'D:':-1,'>:(':-1,
    ':/':-1,':-/':-1,':|':-1,':-|':-1,
    'T_T':-1,';_;':-1,'-_-':-1,
}
_emoticon_re = re.compile(
    '|'.join(re.escape(e) for e in sorted(EMOTICON_POLARITY, key=len, reverse=True))
)

# Emojis (Feature 50)
EMOJI_POSITIVE = set(
    '😀😃😄😁😆😊🙂😍'
    '🥰😘😋😛😜🤗💪'
    '👍👏🎉❤💕💖✅'
    '🌟⭐😎🥳🔥💯👌✨'
)
EMOJI_NEGATIVE = set(
    '😢😭😞😔😟😕🙁☹'
    '😣😖😩😤😡🤬👎💔'
    '😱😨😰😥😓🤮🤢'
    '💩🚫❌😠'
)

print(f'Positivos: {len(POSITIVE_TERMS)} | Negativos: {len(NEGATIVE_TERMS)} | Subjetivos: {len(SUBJECTIVE_TERMS)}')

Positivos: 119 | Negativos: 107 | Subjetivos: 250


## 5. Funcao de Extracao de Features

### Features do artigo incluidas
| Tabela | Features |
|--------|---------|
| Tabela 1 (POS) | 1 ADJ, 3 ADV, 6 DET, 7 INTJ, 9 NUM, 12 PUNCT, 14 SYM, 16 COMP, 17 SUP, 18 X |
| Tabela 2 (Sintaticos) | 19, 20, 21, 22, 23 |
| Tabela 3 (Lexico) | 24, 25, 26, 27, 28, 29, 30, 31 |
| Tabela 5 (Estrutural) | 46 uppercase_ratio |
| Tabela 6 (Twitter) | 47, 48, 49, 50, 51 |

### Features extras
- `neg_before_pos` / `neg_before_neg`: negacao antes de termo positivo/negativo
- `has_negated_positive` / `has_negated_negative`: flags binarias
- `num_intensifiers` / `num_attenuators`: modificadores de intensidade
- `num_all_caps_words`: palavras em maiusculo completo
- `num_exclamations` / `num_questions`: pontuacao expressiva separada
- `repeated_exclamation` / `repeated_question` / `mixed_punct`: repeticao de pontuacao
- `ratio_neg_pos`: ratio negativo/positivo
- `num_first_person`: verbos na primeira pessoa
- `type_token_ratio`: diversidade lexical
- `text_length_chars` / `num_words`: tamanho do texto


In [ ]:
# POS tags selecionados do artigo (Tabela 1: feats 1,3,6,7,9,12,14,18)
POS_SELECTED = {
    'ADJ':   'num_adj',    # Feat 1
    'ADV':   'num_adv',    # Feat 3
    'DET':   'num_det',    # Feat 6
    'INTJ':  'num_intj',   # Feat 7
    'NUM':   'num_num',    # Feat 9
    'PUNCT': 'num_punct',  # Feat 12
    'SYM':   'num_sym',    # Feat 14
    'X':     'num_x',      # Feat 18
}

NEGATION_WINDOW = 3  # janela de tokens apos palavra de negacao


def extract_features(original_text: str, doc) -> dict:
    feats    = {}
    tokens   = [t for t in doc if not t.is_space]
    words    = [t for t in tokens if t.is_alpha]
    n_words  = max(len(words), 1)
    text_low = original_text.lower()

    # == ARTIGO Tabela 1: POS selecionados ========================
    for fname in POS_SELECTED.values():
        feats[fname] = 0
    for t in tokens:
        if t.pos_ in POS_SELECTED:
            feats[POS_SELECTED[t.pos_]] += 1

    # Feat 16 - Comparativos
    feats['num_comp'] = sum(text_low.count(w) for w in COMP_LIST)
    # Feat 17 - Superlativos
    feats['num_sup']  = sum(text_low.count(w) for w in SUP_LIST)

    # == ARTIGO Tabela 2: Padroes sintaticos (19-23) ==============
    p19 = p20 = p21 = p22 = p23 = 0
    for t in doc:
        # Feat 19: ADJ -> NOUN
        if t.pos_ == 'NOUN' and any(w.dep_ == 'amod' and w.pos_ == 'ADJ' for w in t.lefts):
            p19 += 1
        # Feat 20: ADV -> ADJ -> Not(NOUN)
        if t.pos_ == 'VERB':
            has_subj_adj = any(w.dep_ == 'nsubj' and w.pos_ in ('ADJ','PROPN') for w in t.lefts)
            has_adv      = any(w.dep_ in ('advmod','xcomp') and w.pos_ in ('ADV','ADJ') for w in t.rights)
            if has_subj_adj and has_adv:
                p20 += 1
        if t.pos_ == 'ADJ' and t.dep_ == 'ROOT':
            csubj = [w for w in t.rights if w.dep_ == 'csubj' and w.pos_ == 'VERB']
            if csubj and any(w.dep_ == 'xcomp' and w.pos_ == 'ADJ' for v in csubj for w in v.rights):
                p20 += 1
        # Feat 21: ADJ -> ADJ -> Not(NOUN)
        if t.pos_ == 'ADJ' and t.dep_ == 'amod':
            if any(w.pos_ == 'ADJ' and w != t for w in t.head.children):
                p21 += 1
        # Feat 22: NOUN -> ADJ -> Not(NOUN)
        if t.pos_ == 'ADJ' and t.dep_ == 'ROOT':
            if any(w.dep_ == 'nsubj' and w.pos_ == 'NOUN' for w in t.lefts):
                p22 += 1
        # Feat 23: ADV -> VERB + {PCP|GER}
        if t.pos_ == 'VERB' and t.dep_ == 'ROOT':
            formas = [w for w in t.rights
                      if w.pos_ in ('VERB','ADJ')
                      and any(f in w.morph.get('VerbForm',[]) for f in ('Part','Ger'))]
            adv_rt = [w for w in t.children if w.dep_ == 'advmod' and w.pos_ == 'ADV']
            adv_fm = [w for f in formas for w in f.children if w.dep_ == 'advmod']
            if formas and (adv_rt or adv_fm):
                p23 += 1

    feats['pat_adj_noun']          = p19  # Feat 19
    feats['pat_adv_adj_not_noun']  = p20  # Feat 20
    feats['pat_adj_adj_not_noun']  = p21  # Feat 21
    feats['pat_noun_adj_not_noun'] = p22  # Feat 22
    feats['pat_adv_verb_pcp_ger']  = p23  # Feat 23

    # == ARTIGO Tabela 3: Lexico de sentimento (24-31) ============
    n_pos = n_neg = n_subj = n_marks = 0
    for t in words:
        lemma = t.lemma_.lower()
        word  = t.text.lower()
        if lemma in SUBJECTIVE_TERMS  or word in SUBJECTIVE_TERMS:  n_subj  += 1
        if lemma in POSITIVE_TERMS    or word in POSITIVE_TERMS:    n_pos   += 1
        if lemma in NEGATIVE_TERMS    or word in NEGATIVE_TERMS:    n_neg   += 1
        if lemma in OPINION_MARKERS   or word in OPINION_MARKERS:   n_marks += 1

    feats['num_subjective']  = n_subj                        # Feat 24
    feats['num_positive']    = n_pos                         # Feat 25
    feats['num_negative']    = n_neg                         # Feat 26
    feats['num_marks']       = n_marks                       # Feat 27
    feats['prop_subjective'] = round(n_subj  / n_words, 4)  # Feat 28
    feats['prop_positive']   = round(n_pos   / n_words, 4)  # Feat 29
    feats['prop_negative']   = round(n_neg   / n_words, 4)  # Feat 30
    feats['prop_marks']      = round(n_marks / n_words, 4)  # Feat 31

    # == ARTIGO Tabela 5: Feat 46 (uppercase_ratio) ==============
    text_no_sp = re.sub(r'\s', '', original_text)
    n_up_chars = sum(1 for c in original_text if c.isupper())
    feats['uppercase_ratio'] = round(n_up_chars / max(len(text_no_sp), 1), 3)  # Feat 46

    # == ARTIGO Tabela 6: Twitter (47-51) =========================
    feats['num_urls']      = len(re.findall(r'https?://\S+|www\.\S+', original_text, re.I))  # 47
    feats['num_mentions']  = len(re.findall(r'@\w+', original_text))                           # 48
    feats['num_elongated'] = len(re.findall(r'\b\w*(.)\1{2,}\w*\b', original_text))            # 49

    emoji_scores = [1 if c in EMOJI_POSITIVE else -1
                    for c in original_text if c in EMOJI_POSITIVE | EMOJI_NEGATIVE]
    feats['polarity_emojis'] = (                                                               # 50
        round(sum(emoji_scores) / max(len(emoji_scores), 1), 3) if emoji_scores else 0)

    em_scores = [EMOTICON_POLARITY[m] for m in _emoticon_re.findall(original_text) if m in EMOTICON_POLARITY]
    feats['polarity_emoticons'] = (                                                            # 51
        round(sum(em_scores) / max(len(em_scores), 1), 3) if em_scores else 0)

    # == EXTRA: Negacao ==
    neg_before_pos = 0
    neg_before_neg = 0
    for i, t in enumerate(tokens):
        if t.text.lower() in NEGATION_WORDS or t.lemma_.lower() in NEGATION_WORDS:
            window = tokens[i+1 : i+1+NEGATION_WINDOW]
            for wt in window:
                wl = wt.lemma_.lower()
                ww = wt.text.lower()
                if wl in POSITIVE_TERMS or ww in POSITIVE_TERMS:
                    neg_before_pos += 1
                if wl in NEGATIVE_TERMS or ww in NEGATIVE_TERMS:
                    neg_before_neg += 1

    feats['neg_before_pos']       = neg_before_pos
    feats['neg_before_neg']       = neg_before_neg
    feats['has_negated_positive'] = int(neg_before_pos > 0)
    feats['has_negated_negative'] = int(neg_before_neg > 0)

    # == Intensificadores e atenuadores ====================
    n_intens = sum(1 for t in words
                   if t.lemma_.lower() in INTENSIFIERS or t.text.lower() in INTENSIFIERS)
    n_atten  = sum(1 for t in words
                   if t.lemma_.lower() in ATTENUATORS  or t.text.lower() in ATTENUATORS)
    feats['num_intensifiers'] = n_intens
    feats['num_attenuators']  = n_atten

    # == Palavras ALL CAPS (emocao forte) ==================
    word_list = original_text.split()
    all_caps  = [w for w in word_list if w.isupper() and len(w) > 1 and w.isalpha()]
    feats['num_all_caps_words'] = len(all_caps)

    # == Pontuacao expressiva separada =====================
    feats['num_exclamations']     = original_text.count('!')
    feats['num_questions']        = original_text.count('?')
    feats['repeated_exclamation'] = int(bool(re.search(r'!{2,}', original_text)))
    feats['repeated_question']    = int(bool(re.search(r'[?]{2,}', original_text)))
    feats['mixed_punct']          = int(bool(re.search(r'[!?]{2,}', original_text)))

    # == Ratio negativo/positivo ===========================
    feats['ratio_neg_pos'] = round(n_neg / max(n_pos, 1), 4)

    # == Verbos na primeira pessoa =========================
    feats['num_first_person'] = sum(
        1 for t in words
        if t.lemma_.lower() in FIRST_PERSON_VERBS or t.text.lower() in FIRST_PERSON_VERBS
    )

    # == Diversidade lexical e comprimento =================
    unique_lemmas = {t.lemma_.lower() for t in words}
    feats['type_token_ratio']  = round(len(unique_lemmas) / n_words, 4)
    feats['text_length_chars'] = len(original_text)
    feats['num_words']         = len(word_list)

    return feats


# Teste rapido
_s   = 'Produto maravilhoso! Chegou RAPIDO e funcionou perfeitamente. Nao e ruim, e otimo mesmo :-)'
_doc = nlp(_s)
_f   = extract_features(_s, _doc)
print(f'Total de features: {len(_f)}')
print(f'  num_adj={_f["num_adj"]}, num_positive={_f["num_positive"]}, num_negative={_f["num_negative"]}')
print(f'  neg_before_pos={_f["neg_before_pos"]}, neg_before_neg={_f["neg_before_neg"]}')
print(f'  num_all_caps_words={_f["num_all_caps_words"]}, num_exclamations={_f["num_exclamations"]}')
print(f'  type_token_ratio={_f["type_token_ratio"]}, polarity_emoticons={_f["polarity_emoticons"]}')

Total de features: 46
  num_adj=3, num_positive=5, num_negative=1
  neg_before_pos=0, neg_before_neg=1
  num_all_caps_words=1, num_exclamations=1
  type_token_ratio=0.8462, polarity_emoticons=1.0


## 6. Extracao em Batch - Texto e Titulo Separados

O titulo e processado **independentemente** do texto:
- Preserva a funcao sintatica de cada campo
- Um adjetivo no titulo nao interfere nos tokens do texto
- Features do titulo recebem prefixo `title_`


In [ ]:
def extract_features_batch(texts, titles, ids, batch_size=256):
    '''
    Extrai features do texto e do titulo separadamente.
    Colunas do titulo recebem prefixo 'title_'.
    '''
    texts_list  = [str(t) if t is not None else '' for t in texts]
    titles_list = [str(t) if t is not None else '' for t in titles]
    ids_list    = list(ids)
    records     = []

    print(f'Extraindo features de {len(texts_list):,} registros...')

    # Processar em batch (muito mais rapido que loop individual)
    text_docs  = list(nlp.pipe(texts_list,  batch_size=batch_size, n_process=1))
    title_docs = list(nlp.pipe(titles_list, batch_size=batch_size, n_process=1))
    print('  spaCy concluido. Calculando features...')

    for i, (tdoc, tldoc) in enumerate(zip(text_docs, title_docs)):
        # Features do texto (sem prefixo)
        row = extract_features(texts_list[i], tdoc)

        # Features do titulo (com prefixo 'title_')
        title_feats = extract_features(titles_list[i], tldoc)
        for k, v in title_feats.items():
            row[f'title_{k}'] = v

        row['id'] = ids_list[i]
        records.append(row)

        if (i + 1) % 5_000 == 0:
            print(f'  {i+1:,} / {len(texts_list):,}')

    print('Concluido!')
    df_out = pd.DataFrame(records)
    cols = ['id'] + [c for c in df_out.columns if c != 'id']
    return df_out[cols]


# Extrair features de treino
train_feats = extract_features_batch(train_df['text'], train_df['title'], train_df['id'])
train_feats.to_csv(current_dir / 'features_train.csv', index=False)

print(f'\nShape das features de treino: {train_feats.shape}')
n_text  = len([c for c in train_feats.columns if not c.startswith('title_') and c != 'id'])
n_title = len([c for c in train_feats.columns if c.startswith('title_')])
print(f'  Features do texto : {n_text}')
print(f'  Features do titulo: {n_title}')
print(f'  Total             : {n_text + n_title}')
print(f'\nColunas: {list(train_feats.columns)}')

Extraindo features de 41,005 registros...
  spaCy concluido. Calculando features...
  5,000 / 41,005
  10,000 / 41,005
  15,000 / 41,005
  20,000 / 41,005
  25,000 / 41,005
  30,000 / 41,005
  35,000 / 41,005
  40,000 / 41,005
Concluido!

Shape das features de treino: (41005, 93)
  Features do texto : 46
  Features do titulo: 46
  Total             : 92

Colunas: ['id', 'num_adj', 'num_adv', 'num_det', 'num_intj', 'num_num', 'num_punct', 'num_sym', 'num_x', 'num_comp', 'num_sup', 'pat_adj_noun', 'pat_adv_adj_not_noun', 'pat_adj_adj_not_noun', 'pat_noun_adj_not_noun', 'pat_adv_verb_pcp_ger', 'num_subjective', 'num_positive', 'num_negative', 'num_marks', 'prop_subjective', 'prop_positive', 'prop_negative', 'prop_marks', 'uppercase_ratio', 'num_urls', 'num_mentions', 'num_elongated', 'polarity_emojis', 'polarity_emoticons', 'neg_before_pos', 'neg_before_neg', 'has_negated_positive', 'has_negated_negative', 'num_intensifiers', 'num_attenuators', 'num_all_caps_words', 'num_exclamations', 'n

## 7. Preparar Dados de Treino

In [ ]:
merged = train_df[['id', 'rating']].merge(train_feats, on='id')
FEATURE_COLS = [c for c in merged.columns if c not in ('id', 'rating')]

X = merged[FEATURE_COLS].values
y = merged['rating'].values

print(f'X: {X.shape} | y: {y.shape}')
print(f'Classes: {sorted(set(y))}')
print(pd.Series(y).value_counts().sort_index())


X: (41005, 92) | y: (41005,)
Classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
1    8205
2    8178
3    8194
4    8198
5    8230
Name: count, dtype: int64


## 8. Treinar GBT - 5-Fold Stratified Cross-Validation

Modelo: **LightGBM (GBT)** - melhor classificador segundo o artigo.
Metrica: **macro F1** com **5-fold estratificado**.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []
models    = []
oof_preds = np.zeros(len(y), dtype=int)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators      = 2000,
        learning_rate     = 0.03,
        num_leaves        = 127,
        max_depth         = 8,
        min_child_samples = 30,
        subsample         = 0.8,
        colsample_bytree  = 0.8,
        reg_alpha         = 0.1,
        reg_lambda        = 1.0,
        class_weight      = 'balanced',
        random_state      = 42,
        n_jobs            = -1,
        verbose           = -1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set  = [(X_val, y_val)],
        callbacks = [lgb.early_stopping(100, verbose=False),
                     lgb.log_evaluation(period=0)],
    )
    preds = model.predict(X_val)
    oof_preds[val_idx] = preds
    f1 = f1_score(y_val, preds, average='macro')
    f1_scores.append(f1)
    models.append(model)
    print(f'Fold {fold+1}/5  best_iter={model.best_iteration_:4d}  F1 macro={f1:.4f}')

print(f'\nCV F1 macro: {np.mean(f1_scores):.4f} +/- {np.std(f1_scores):.4f}')
print('\nClassification report (OOF):')
print(classification_report(y, oof_preds))


Fold 1/5  best_iter= 295  F1 macro=0.4457
Fold 2/5  best_iter= 302  F1 macro=0.4550
Fold 3/5  best_iter= 292  F1 macro=0.4562
Fold 4/5  best_iter= 384  F1 macro=0.4624
Fold 5/5  best_iter= 283  F1 macro=0.4561

CV F1 macro: 0.4551 +/- 0.0054

Classification report (OOF):
              precision    recall  f1-score   support

           1       0.50      0.60      0.54      8205
           2       0.36      0.33      0.35      8178
           3       0.36      0.32      0.34      8194
           4       0.42      0.37      0.39      8198
           5       0.62      0.69      0.65      8230

    accuracy                           0.46     41005
   macro avg       0.45      0.46      0.46     41005
weighted avg       0.45      0.46      0.46     41005



## 9. Gerar Predicoes e Submission

In [ ]:
test_feats = extract_features_batch(test_df['text'], test_df['title'], test_df['id'])
test_feats.to_csv(current_dir / 'features_test.csv', index=False)

for col in FEATURE_COLS:
    if col not in test_feats.columns:
        test_feats[col] = 0
X_test = test_feats[FEATURE_COLS].values

all_preds  = np.array([m.predict(X_test) for m in models])
preds_test = stats.mode(all_preds, axis=0).mode.flatten()

sample_sub           = pd.read_csv(csv_folder / 'sample_submission.csv')
sample_sub['rating'] = preds_test
sample_sub.to_csv(current_dir / 'submission.csv', index=False)

print(f'Submission salva: {len(sample_sub):,} predicoes.')
print('\nDistribuicao das predicoes:')
print(pd.Series(preds_test).value_counts().sort_index())


Extraindo features de 10,252 registros...
  spaCy concluido. Calculando features...
  5,000 / 10,252
  10,000 / 10,252
Concluido!
Submission salva: 10,252 predicoes.

Distribuicao das predicoes:
1    2508
2    1851
3    1781
4    1740
5    2372
Name: count, dtype: int64
